# Comparing representations: whole days or individual timesteps?

Clustering decides **which days belong together**. Representation decides **which values
stand in for those days**. These are separate choices: changing the representation can leave
the grouping untouched while changing its representative profile.

This tutorial compares the four basic rules — `mean`, `medoid`, `maxoid`, and
`minmax_mean` — on four small days whose values we can check by hand. We will distinguish
selecting a **whole day** from computing values separately for each **attribute and
timestep**.

The companion [Duration representations](duration_representations.ipynb) uses a full year
of hourly data to compare these baselines with duration-curve fitting, local/global scope,
extreme preservation, and concurrency ordering. For the underlying algorithms, see
[Representation](../explanation/how-aggregation-works/03_representation.ipynb).

## 1  A cluster to compare on

We aggregate the [tiny six-day dataset](../explanation/how-aggregation-works/01_preprocessing.ipynb)
into two clusters. We follow its largest cluster: four days, each with four timesteps and
two attributes. It contains both ordinary days and the series' 10 MW peak.

Rescaling is disabled so we can examine each rule's raw output. With the basic rules,
each cluster's representative is selected from, or constructed using, that cluster's members.
The maxoid also compares candidate members with days outside the cluster when scoring them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, MinMaxMean
from tsam.metrics import series_statistics
from tsam.plot import path_panels

pio.renderers.default = "notebook_connected"
ATTRS = ["solar", "load"]
UNITS = {"solar": "W/m²", "load": "MW"}
tiny = pd.read_csv("../data/tiny.csv", index_col=0, parse_dates=True)

baseline = tsam.aggregate(
    tiny,
    n_clusters=2,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    preserve_column_means=False,
)
N_TIMESTEPS = baseline.n_timesteps_per_period
focus = max(baseline.cluster_counts, key=baseline.cluster_counts.get)
members = np.flatnonzero(baseline.cluster_assignments == focus)
print(f"The cluster we follow contains days {members.tolist()}.")

These are the numbers every rule must condense into one profile. Columns distinguish the
attribute and the timestep; rows distinguish whole days.

In [ ]:
members_physical = tsam.unstack_to_periods(tiny, period_duration="1D").iloc[members]
members_physical.index = [f"day{day}" for day in members]
members_physical

## 2  The cluster, drawn

A period is not a point — it is a **path**. With two attributes we can draw it directly: solar
on one axis, load on the other, one marker per timestep. Each day traces a path from `t0` to
`t3`, and the arrows keep the direction of travel visible.

Each day has a fixed colour, identified in the legend. Its own panel highlights it; the other
members appear in their own colours, faded for context. **The day colours stay the same in
the representative comparison below.** Click a legend entry to hide or show that day across
all panels. All panels use the same axis limits. Hover over a marker for its day, timestep,
and values.

Read the four days as a family: all start at `solar = 0`, move right, and return to
`solar = 0` by `t3`. **day5** is the outlier — it barely leaves the load axis and climbs to
10 MW. Separating the panels keeps shared points visible without moving any data.

In [ ]:
member_paths = {
    f"day{day}": tiny.iloc[day * N_TIMESTEPS : (day + 1) * N_TIMESTEPS]
    for day in members
}
DAY_COLORS = {
    "day2": "#0072B2",
    "day3": "#D55E00",
    "day4": "#009E73",
    "day5": "#8759A5",
}
REPRESENTATIVE_COLOR = "#222222"

path_panels(
    member_paths,
    "solar",
    "load",
    background=member_paths,
    units=UNITS,
    colors=DAY_COLORS,
    title="Four member days · arrows follow t0 → t1 → t2 → t3",
).show()

## 3  What does each rule compare or combine?

A whole day is one vector containing **all attributes at all timesteps**. Here it has eight
coordinates: four solar values followed by four load values. Distances use the normalized,
weighted vectors from preprocessing, so the physical units in our plots are not the units
used to compare distances.

| Rule | Unit of selection or calculation | How the representative is obtained |
|---|---|---|
| `medoid` | **Whole day** | Among the cluster's members, select the day with the smallest summed Euclidean distance to the other members. Copy its entire path. |
| `maxoid` | **Whole day** | Among the cluster's members, select the day with the largest summed Euclidean distance to **all days in the dataset**. Copy its entire path. |
| `mean` | **One attribute at one timestep** | Average that coordinate across the member days, then repeat for every coordinate. |
| `minmax_mean` | **One attribute at one timestep** | Take a min, max, or mean across member days at the same timestep, according to the choice for that attribute. |

For `mean` and `minmax_mean`, the calculation at `t1` uses only the members' `t1` values;
it does not pool values from `t0`, `t2`, or `t3`. For medoid and maxoid there is one selection
for the **entire day**, never a separate winning day for each timestep or attribute.

The mean is also the vector centroid of the days. Being a centroid does not make it an
observed day: its coordinates are still averages calculated separately. For `minmax_mean`
we request maximum load at each timestep and mean solar generation.

In [ ]:
rules = {
    "mean": "mean",
    "medoid": "medoid",
    "maxoid": "maxoid",
    "minmax_mean": MinMaxMean(max_columns=["load"], min_columns=[]),
}
profiles = {}
for name, representation in rules.items():
    result = tsam.aggregate(
        tiny,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )
    cid = int(result.cluster_assignments[members[0]])
    assert set(np.flatnonzero(result.cluster_assignments == cid)) == set(members)
    profiles[name] = result.cluster_representatives.loc[cid]

pd.concat(profiles, names=["rule"]).round(2)

### Follow one timestep

At `t1`, the four solar values are 3, 2, 1, and 1: their mean is **1.75**. The load values
are 5, 4, 6, and 7: their mean is **5.5**. Thus the mean representative's point is
`(solar=1.75, load=5.5)`, which no member has at `t1`.

`minmax_mean` takes the same mean solar value but the maximum load, **7**. Its solar/load
pair combines two separate calculations. At other timesteps or with other min/max columns,
the selected extremes may come from different days.

In [ ]:
t1 = pd.DataFrame({name: frame.iloc[1] for name, frame in member_paths.items()}).T
t1.loc["mean"] = profiles["mean"].iloc[1]
t1.loc["minmax_mean"] = profiles["minmax_mean"].iloc[1]
t1

## 4  What each rule keeps

Compare the minimum, mean, and maximum in physical units. The first row pools the four
member days; the remaining rows describe the four representative profiles. Every timestep
has equal duration, so the mean also tells us about the total over a representative day.

In [ ]:
series_statistics(
    {"cluster members": pd.concat(member_paths.values()), **profiles}
).round(2)

* **`mean` keeps both attribute means**, but lowers the load peak from 10 to 7 MW.
* **`medoid` copies day4**. Its paired values and order are observed, but its means are the
  means of that one day, and it misses the 10 MW peak.
* **`maxoid` copies day5**, which happens to contain the peak here. Globally large distance
  does **not** guarantee retaining every attribute's minimum or maximum on other datasets.
* **`minmax_mean` keeps solar's mean and takes an envelope on load**. Its load values happen
  to match day5 here, but its solar values do not: the whole profile is constructed.

Selecting a real day preserves that day's internal pairings; it does not promise that the
reconstructed dataset has the original dataset's correlations or totals. Constructed
profiles can preserve selected statistics without being days that actually occurred.

## 5  Real days land on a member's whole path

Each panel shows one representative as a **black dashed path**. The reference days retain
their colours from section 2 and are identified in the legend. Coloured rings remain visible
where representative and member points coincide. Follow the arrows and `t0`–`t3` labels:
a whole-day match must reproduce **every timestep and attribute**, not just visit the same
points.

The medoid retraces **day4 (green)** and the maxoid retraces **day5 (purple)**. Mean and
min/max/mean follow paths assembled from coordinate-wise calculations.

In [ ]:
path_panels(
    profiles,
    "solar",
    "load",
    background=member_paths,
    units=UNITS,
    colors=DAY_COLORS,
    path_color=REPRESENTATIVE_COLOR,
    path_label="representative",
    dash="dash",
    title="Four basic rules · whole days or constructed paths",
).show()

## 6  From individual days to distributions

The basic choice is what the representative must retain: an observed path, a coordinate-wise
average, or an envelope. A further option is to approximate **how often values occur**.
Distribution representations pool each attribute's values across member timesteps, fit a
duration curve, and then assign those fitted values to a new time order. They belong to the
constructed-profile family, but use a different calculation from a same-timestep mean.

Continue with **[Duration representations](duration_representations.ipynb)** for a full year
of hourly data, comparisons with these basic rules, and separate choices about fitting and
ordering.

These examples disable tsam's default rescaling (`preserve_column_means=True`). Rescaling
can change both observed and constructed profiles; assess the final configuration you intend
to use. See [Rescaling](../explanation/how-aggregation-works/05_rescaling.ipynb).

* [Representation](../explanation/how-aggregation-works/03_representation.ipynb) — the algorithms worked by hand.
* [Representations how-to](../how-to/representations.ipynb) — configuration recipes.
* [Extreme periods](../explanation/how-aggregation-works/04_extreme_periods.ipynb) — retain a specifically chosen extreme period.